# S2 — Бэктест и калибровка proxy §8.10 (CoinGecko)

**Цель** (`upgrade_plan.md` **S2**): на историческом ряду проверить простые правила по `composite_onchain` / подобрать пороги и веса **до** фиксации продакшен-настроек (см. `plan.md` §8.10, п. 9 composite, п. 10 сигналы и backtest).

- Данные и формулы совпадают с `bit_trend.data.coingecko_onchain` (один запрос `market_chart`).
- **Загрузите `.env` до импорта модуля**, если калибруете веса из `COMPOSITE_810_*` — иначе возьмутся значения по умолчанию.
- Нужен доступ в интернет к API CoinGecko.

## 1. Импорты и пути

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

root = Path("..").resolve()
sys.path.insert(0, str(root))

try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
except ImportError:
    pass

from bit_trend.data.coingecko_onchain import get_coingecko_810_dataframe

warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Загрузка ряда §8.10

In [ ]:
df = get_coingecko_810_dataframe()
if df is None or df.empty:
    raise RuntimeError("Не удалось получить данные CoinGecko — проверьте сеть и USE_COINGECKO_ONCHAIN.")

df = df.sort_index()
print(df.index.min(), "→", df.index.max(), "rows:", len(df))
display(df[["price", "composite_onchain", "mvrv_z", "nupl_z", "sopr_z"]].tail())

## 3. Композит с произвольными весами и бэктест long-only

Правила как в `plan.md` §8.10: **вход в BTC**, если сигнал **&lt; buy_th**; **выход в кэш**, если **&gt; sell_th** (гистерезис). Сигнал на конец дня \(t\) исполняется с **дневной доходностью** начиная с \(t+1\) (без подглядывания в ту же свечу).

Учитывается односторонняя комиссия `fee` при каждой смене позиции.

In [ ]:
def composite_weighted(
    d: pd.DataFrame,
    w_mvrv: float,
    w_nupl: float,
    w_sopr: float,
    w_dd: float,
    w_vol: float,
) -> pd.Series:
    return (
        w_mvrv * d["mvrv_z"]
        + w_nupl * d["nupl_z"]
        + w_sopr * d["sopr_z"]
        + w_dd * (-d["drawdown_z"])
        + w_vol * d["volatility_z"]
    )


def smooth_series(s: pd.Series, win: int) -> pd.Series:
    if win <= 1:
        return s
    return s.rolling(win, min_periods=1).mean()


def hysteresis_positions(signal: pd.Series, buy_th: float, sell_th: float) -> pd.Series:
    """signal[i] известен на конец дня i; позиция pos[i] — удержание BTC в течение дня i (0=cash, 1=BTC)."""
    pos = np.zeros(len(signal), dtype=np.int8)
    p = 0
    for i in range(len(signal) - 1):
        s = signal.iloc[i]
        if pd.isna(s):
            p = p
        elif p == 0 and s < buy_th:
            p = 1
        elif p == 1 and s > sell_th:
            p = 0
        pos[i + 1] = p
    return pd.Series(pos, index=signal.index)


def run_backtest(
    d: pd.DataFrame,
    composite: pd.Series,
    *,
    buy_th: float = -1.0,
    sell_th: float = 1.0,
    smooth: int = 7,
    fee: float = 0.0005,
) -> dict:
    price = d["price"].astype(float)
    ret = price.pct_change()
    sig = smooth_series(composite, smooth)
    pos = hysteresis_positions(sig, buy_th, sell_th)
    pos = pd.Series(pos, index=d.index).astype(float)

    strat_ret = pos * ret
    turnover = (pos.diff().abs() > 0).fillna(False)
    costs = turnover.astype(float) * fee
    equity_curve = (1 + strat_ret - costs).cumprod()
    equity_curve = equity_curve.replace([np.inf, -np.inf], np.nan).dropna()

    bh = (1 + ret.fillna(0)).cumprod()
    bh = bh.loc[equity_curve.index]

    r = strat_ret.loc[equity_curve.index] - costs.loc[equity_curve.index]
    r = r.iloc[1:].dropna()
    sharpe = 0.0
    if len(r) > 10 and r.std() > 0:
        sharpe = float(r.mean() / r.std() * np.sqrt(252))

    ec = equity_curve.clip(lower=1e-8)
    roll_max = ec.cummax()
    dd = (ec / roll_max - 1).min()

    return {
        "equity": equity_curve,
        "buy_hold": bh,
        "pos": pos,
        "total_return": float(ec.iloc[-1] - 1),
        "buy_hold_return": float(bh.iloc[-1] - 1),
        "max_drawdown": float(dd),
        "sharpe": sharpe,
        "n_turns": int(turnover.sum()),
    }


def train_test_split_idx(d: pd.DataFrame, train_frac: float = 0.7):
    n = len(d)
    cut = int(n * train_frac)
    return d.index[:cut], d.index[cut:]

## 4. Базовый прогон (веса как в текущем `.env` / модуле)

Колонка `df["composite_onchain"]` уже посчитана в `get_coingecko_810_dataframe()` с теми `COMPOSITE_810_*`, что были в окружении **при первом импорте** `coingecko_onchain`.

In [ ]:
baseline = run_backtest(df, df["composite_onchain"], buy_th=-1.0, sell_th=1.0, smooth=7)
print(
    "Стратегия / buy-hold:",
    f"{baseline['total_return']:.2%}",
    f"{baseline['buy_hold_return']:.2%}",
)
print("Sharpe (дн.):", round(baseline["sharpe"], 3), "MaxDD:", f"{baseline['max_drawdown']:.2%}", "Сделок:", baseline["n_turns"])

## 5. Сетка порогов (in-sample на первых 70% дней)

Калибровка **не подменяет** валидацию на новых данных; она лишь сужает разумный диапазон для ручной проверки и записи в `.env`.

In [ ]:
tr_idx, te_idx = train_test_split_idx(df, 0.7)
df_tr = df.loc[tr_idx]
df_te = df.loc[te_idx]
comp_tr = df_tr["composite_onchain"]
comp_te = df_te["composite_onchain"]

buy_grid = [-1.5, -1.25, -1.0, -0.75, -0.5]
sell_grid = [0.5, 0.75, 1.0, 1.25, 1.5]
rows = []
for b in buy_grid:
    for s in sell_grid:
        if b >= s:
            continue
        out = run_backtest(df_tr, comp_tr, buy_th=b, sell_th=s, smooth=7)
        rows.append({"buy_th": b, "sell_th": s, "ret_tr": out["total_return"], "sharpe_tr": out["sharpe"], "mdd_tr": out["max_drawdown"]})
grid_df = pd.DataFrame(rows).sort_values("sharpe_tr", ascending=False)
display(grid_df.head(12))

best = grid_df.iloc[0]
oob = run_backtest(df_te, comp_te, buy_th=best["buy_th"], sell_th=best["sell_th"], smooth=7)
print("Лучшие порога (train):", float(best["buy_th"]), float(best["sell_th"]))
print("OOS total return:", f"{oob['total_return']:.2%}", "Sharpe:", round(oob["sharpe"], 3))

## 6. Сетка весов (нормализация положительной части до 1; волатильность отдельно)

Набор кандидатов — как в примерах плана; при необходимости расширьте список `candidates`.

In [ ]:
def norm_weights(w_m, w_n, w_s, w_d):
    t = w_m + w_n + w_s + w_d
    return w_m / t, w_n / t, w_s / t, w_d / t


candidates = [
    (0.30, 0.25, 0.20, 0.25, -0.10),
    (0.35, 0.25, 0.15, 0.25, -0.10),
    (0.25, 0.30, 0.20, 0.25, -0.15),
    (0.40, 0.20, 0.20, 0.20, -0.05),
]

w_rows = []
for w_m, w_n, w_s, w_d, w_v in candidates:
    nm, nn, ns, nd = norm_weights(w_m, w_n, w_s, w_d)
    comp = composite_weighted(df_tr, nm, nn, ns, nd, w_v)
    out = run_backtest(df_tr, comp, buy_th=-1.0, sell_th=1.0, smooth=7)
    w_rows.append(
        {
            "wm": nm,
            "wn": nn,
            "ws": ns,
            "wd": nd,
            "wv": w_v,
            "ret": out["total_return"],
            "sharpe": out["sharpe"],
        }
    )

wgrid = pd.DataFrame(w_rows).sort_values("sharpe", ascending=False)
display(wgrid)

br = wgrid.iloc[0]
comp_full = composite_weighted(df, br["wm"], br["wn"], br["ws"], br["wd"], br["wv"])
print("Пример строк для .env (подставьте после ручной проверки):")
print(f"COMPOSITE_810_W_MVRV={br['wm']:.4f}")
print(f"COMPOSITE_810_W_NUPL={br['wn']:.4f}")
print(f"COMPOSITE_810_W_SOPR={br['ws']:.4f}")
print(f"COMPOSITE_810_W_DRAWDOWN={br['wd']:.4f}")
print(f"COMPOSITE_810_W_VOLATILITY={br['wv']:.4f}")

## 7. График: цена и composite (plotly)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    _comp = comp_full
except NameError:
    _comp = df["composite_onchain"]
cs = smooth_series(_comp, 7)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df.index, y=df["price"], name="BTC price", line=dict(color="black")), secondary_y=False)
fig.add_trace(go.Scatter(x=df.index, y=cs, name="composite (7d)", line=dict(color="royalblue")), secondary_y=True)
fig.add_hline(y=-1.0, line_dash="dash", secondary_y=True, opacity=0.4)
fig.add_hline(y=1.0, line_dash="dash", secondary_y=True, opacity=0.4)
fig.update_layout(height=420, title="CoinGecko proxy §8.10 — калибровка S2", legend=dict(yanchor="top", y=0.99))
fig.update_yaxes(title_text="USD", secondary_y=False)
fig.update_yaxes(title_text="Composite z", secondary_y=True)
fig.show()

## 8. Что перенести в прод

1. Выбранные **веса** — в `.env` как `COMPOSITE_810_W_*` (см. `.env.example`).
2. Если используете `composite` внутри `BitTrendScorer`, см. `SCORER_WEIGHT_COMPOSITE_810` и `SCORER_COMPOSITE_810_SCALE`.
3. Порога входа/выхода из этого ноутбука — **логика исследования**; в приложении сигналы по-прежнему задаёт `BitTrendScorer`, пока явно не связаны с порогами composite.